# Networks of persons in relation to organisations


Cf. [this notebook](https://github.com/Sciences-historiques-numeriques/histoire_numerique_methodes/blob/main/analyse_reseaux/reseaux_florence.ipynb) about families and power in Renaissance Florence for an introduction to network analysis with the *networkx* Python library



In [ ]:
import pandas as pd


import networkx as nx
from networkx.algorithms import bipartite

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.ticker import MaxNLocator

from scipy.stats import spearmanr
import statsmodels.api as sm

import numpy as np
import seaborn as sns
import math
import os

In [ ]:
### this library allows to do SQL queries on dataframes
import duckdb

In [ ]:
### Librairies déjà installées avec Python
import pprint
import csv

import sqlite3 as sql
import pickle

import time
import datetime
from dateutil import parser


from shutil import copyfile


In [ ]:
### Importer un module de fonctions crées ad hoc
##  ATTENTION : le fichier 'sparql_functions.py' doit se trouver 
#   dans un dossier qui se situe dans le chemin ('path') de recherche
#   vu par le présent carnet Jupyter afin que
#   l'importation fonctionne correctement

import sys
from importlib import reload

# Add parent directory to the path
sys.path.insert(0, '..')

### If you want to add the parent-parent directory,
sys.path.insert(0, '../..')



In [ ]:

import network_analysis_functions as naf
import bivariate_library as bl

In [ ]:
print(reload(bl))  

In [ ]:
import warnings
warnings.filterwarnings('ignore')


## Get and inspect the data


We use in this notebook the network data prepared in these two files:
* Get the [relationships with SPARQL](https://github.com/Sciences-historiques-numeriques/astronomers/blob/main/documentation/wikidata/data-analysis/da6-person-organisation-network.md)
* Prepare the [SQL view and export the data to CSV](https://github.com/Sciences-historiques-numeriques/astronomers/blob/main/documentation/wikidata/data-analysis/da6-person-organisation-network.sql)


Regarding persons, we reuse the file with the persons, their periods of activity and coded countries, that we created in the correspondence analysis chapter.




### Get the network data

In [ ]:
### Import information about relations to organisations - df_po

csv_address='da_data/da6-persons-organisations-relations.csv'
df_po = pd.read_csv(csv_address)
df_po.head(3)

In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_po.info()

In [ ]:
### Number of types of relationships
df_rela_type=pd.DataFrame(df_po.groupby(by='relationship').size())
df_rela_type.columns=['number']
df_rela_type

### Get the data about persons

#### First: get the individuals (20000) used for correspondence analysis 


In [ ]:
### Information about persons with gender, birth date,
# coded country, etc.
csv_address='da_data/da4-AFC.csv'
### csv_address='da_data/da2-birth-place.csv'
df_p_AFC = pd.read_csv(csv_address)
df_p_AFC.head(3)

In [ ]:
df_p_AFC=df_p_AFC[['uriPer', 'labelPer', 'birthYear', 'gender', 'labelPlace', 'REGION', 'NAME_ENGL', 'coded_country', 'periodsActivity']]

In [ ]:
### Rename columns to have shorter labels
df_p_AFC.columns=['person_uri',
 'labelPer',
 'birthYear',
 'gender',
 'labelPlace',
 'REGION',
 'NAME_ENGL',
 'country',
 'per_activ'
 ]

In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_p_AFC.info()

#### Second: get the individuals (8000) used for clustering


In [ ]:
file_address='da_data/da5-MCA-clusters.csv'
df_p_MCA = pd.read_csv(file_address)
df_p_MCA.head(3)

In [ ]:
### Rename columns to have shorter labels
df_p_MCA.columns=['person_uri',
 'labelPer',
 'birthYear',
 'gender',
 'labelPlace',
 'geometry',
 'REGION',
 'NAME_ENGL',
 'country',
 'per_activ',
 'pk_person_features',
 'occup_m',
 'occup_s',
 'empl']

In [ ]:
df_p_MCA=df_p_MCA[['person_uri',
 'occup_m',
 'occup_s',
 'empl']
]

In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_p_MCA.info()

#### Third: add the additional NCA features to the first dataframe

In [ ]:
df_p=df_p_AFC.merge(df_p_MCA, left_on=['person_uri'], right_on=['person_uri'], how='left')
df_p=df_p.fillna('')

In [ ]:
print(len(df_p))
df_p.iloc[:3]

We can observe the limits a restriction to persons that only have the additional features: 

if we restrict to persons with MCA features, Albert Einstein would not be taken into account in the analysis of the network!

We cannot proceed with adding insights from clustering to the network: additional information should be first added to our information system.

We could just add the main occupation, with additional preparation (not yet implemented).


In [ ]:
### Find Albert Einstein
duckdb.query("""
select *
from df_p
where labelPer like '%Einst%'
""").to_df()

## Inspect the features of the relationships

### Distribution of relationships per country

#### Writing SQL queries with the duckdb library

In [ ]:
### Distribution of coded countries
# using the duckdb library and a SQL query

# the result is the same as in the cell below
dfc=duckdb.query("""
    SELECT country, COUNT(*) as num
    FROM df_p
    GROUP BY country
    ORDER BY num DESC
  """).to_df()
dfc

In [ ]:
### Distribution of coded countries
dfc=pd.DataFrame(df_p.groupby(by=['country']).size())
dfc.sort_values(by=0,ascending=False)


## Merge the two dataframes

This will restrict the information about relations to the persons that are present in the file with the individuals (*da4-AFC.csv* with additional information from *da5-MCA-clusters.csv*)

In [ ]:
df_p=df_p[['person_uri', 'labelPer', 'gender', 'birthYear','country', 'per_activ']]
df_p.head(3)

In [ ]:
### Merge the two dataframes:
# only relations that involve persons in df_p
# will remain
df_rel=df_po.merge(df_p,on='person_uri')
print(len(df_rel))
df_rel.head(2)

### Inspect the merge result

In [ ]:
### Unique persons
unique_p=pd.DataFrame(df_rel['person_uri'].unique())
unique_p.columns=['person_uri']
len_up=len(unique_p)
print(f"Number of persons with relations: {len_up}")
unique_p.head(2)

In [ ]:
### Number of types of relationships and per person proportion
df_rela_type=pd.DataFrame(df_rel.groupby(by='relationship').size())
df_rela_type.columns=['number']
df_rela_type['propr_rela_per_person']=df_rela_type['number'].apply(lambda x : x/len_up)
df_rela_type

### Restriction to persons with memberships

We observe from the former queries, that some persons have no memberships. 

In our analysis of the field, notably with a question about correlation  we decide 


In [ ]:
###  Count number of relations per person
df_rel = duckdb.query("""
    with tw1 AS (
    -- we create a list of persons with memberships
    SELECT DISTINCT person_uri
    FROM df_rel
    WHERE relationship='membership'
    )
    SELECT *
    FROM df_rel 
    WHERE person_uri IN (SELECT person_uri FROM tw1)
    
""").to_df()
print(len(df_rel))
df_rel.iloc[:2]

In [ ]:
###  Count number of relations per person
duckdb.query("""
    WITH tw1 AS (
    -- we create a list of unique persons
    SELECT DISTINCT person_uri
    FROM df_rel
    )
    SELECT relationship, COUNT(*) as number, 
            ROUND(COUNT(*)/(SELECT COUNT(*) FROM tw1),2) AS propr_rela_per_person
    FROM df_rel 
    GROUP BY relationship
    ORDER BY relationship    
""").to_df()

### Number of relationships per person

In [ ]:
###  Count number of relations per person
print('Number of relations per person:')
df_pn = duckdb.query("""
    with tw1 AS (
    SELECT person_uri, COUNT(*) as number
    FROM df_rel
    GROUP BY person_uri
    )
    SELECT labelPer,number, birthYear,country,per_activ, tw1.person_uri
    from tw1 join df_p on df_p.person_uri = tw1.person_uri
    ORDER BY number DESC
""").to_df()
df_pn.iloc[:8]

In [ ]:
### Statistical profile
print(df_pn.number.describe())

In [ ]:
# Plot the distribution number of relations to organisations

min_val = df_pn.number.min()
max_val = df_pn.number.max()

plt.figure(figsize=(12, 3))
ax = sns.violinplot(data=df_pn.number, orient='h')

# Set the y-axis limits to exactly match the data range
ax.set_xlim(min_val, max_val)

plt.title('Distribution (density) of relations of persons to organisations')
plt.show()

### Inspect the persons with a high number of relationships

In [ ]:
### Persons that have 10 or more relations to organisations
print(len(df_pn[df_pn.number > 9]))

In [ ]:
# Plot the distribution number of relations to organisations

df=df_pn[df_pn.number > 9]

min_val = df.number.min()
max_val = df.number.max()

plt.figure(figsize=(12, 3))
ax = sns.violinplot(data=df.number, orient='h')

# Set the y-axis limits to exactly match the data range
ax.set_xlim(min_val, max_val)

plt.title('Distribution (density) of relations of persons to organisations')
plt.show()

In [ ]:
### Number of types of relationships, all the persons
df_rela_type=pd.DataFrame(df_rel.groupby(by='relationship').size())
df_rela_type.columns=['number']

### Type of relationship of the 700 more connected persons (merge on the fly)
df_pn[df_pn.number > 9].merge(df_rel, left_on='person_uri', right_on='person_uri').groupby(by='relationship').size()

df_rela_type=df_rela_type.join((df_pn[df_pn.number > 9].\
                   merge(df_rel, left_on='person_uri', right_on='person_uri').groupby(by='relationship').size()).rename('rel_700'))

### Add the proportion between all the relations and those of the most connected persons
df_rela_type['prop']=df_rela_type.apply(lambda x: x['rel_700']/x['number'], axis=1)
### We observe that memberships are overrepresented for this very well known persons
df_rela_type


## Relationships with organisations in relation to activity periods

Observe if periods are specific to relations and which ones 

In [ ]:
### Distribution of RELATIONS by periods of activity
activities_per = pd.DataFrame(df_rel.groupby(by='per_activ').size())
activities_per.columns=['number_rel']
print(activities_per)

In [ ]:
### Distribution of PERSONS by periods of activity
period_per = pd.DataFrame(df_p.groupby(by='per_activ').size())
period_per.columns=['number_per']
print(period_per)

In [ ]:
df_ap_per=activities_per.join(period_per)

### Number of activities divided by number of persons
df_ap_per['proportion']=df_ap_per.apply(lambda x: x.number_rel/x.number_per, axis=1).round(3)
df_ap_per

In [ ]:
### Plot the evolution in time


ax = df_ap_per.proportion.plot(kind='bar',rot=50, fontsize=10, figsize=(4,3))
ax.bar_label(ax.containers[-1], size=8)


current_min, current_max = ax.get_ylim()
ax.set_ylim(current_min, current_max+0.3)

# Rotate labels 60 degrees and reduce font size
plt.xticks(rotation=60, fontsize=8)

plt.ylabel('Number')
plt.xlabel('Periods')
plt.title('Distribution of relationships in relation to activity periods')
plt.show()

In [ ]:
### Create a cross-table
crosstab = pd.crosstab(df_rel['per_activ'],df_rel['relationship'])
# Optional: Remove the index name if you want a cleaner look
# crosstab.index.name = None
crosstab.columns.name = None
crosstab


In [ ]:
bl.check_chi_square_test_validity(crosstab)

In [ ]:
expected=bl.bivariate_stats(crosstab)

#### Comment

A relation appears between type of relationship and period, but it is weak. As we can observe in the heatmap of adjusted residuals: membership tends to be proportionally more present in the 19th century, while study and employment are more present in the 20th century. 

In [ ]:
pp = bl.plot_chi2_residuals(crosstab.T, figsize=(8, 2.5))

## Relationships with organisations in relation to periods: analysis per relationship type

### Universities - education

In [ ]:
### Create a cross-table for inspection
df=df_rel[df_rel['relationship']=='education']
# Create the crosstab
crosstab = pd.crosstab(df['organisation_label'],df['per_activ'])

# Optional: Remove the index name if you want a cleaner look
crosstab.index.name = None
# crosstab.columns.name = None

crosstab=crosstab.fillna(0)
crosstab=crosstab.astype(int)
crosstab['sum'] = crosstab.sum(axis=1)

In [ ]:
### Evolution of number of students in major universities (population in Wikidata)
print('Number of persons per organisation and period:', len(crosstab))
## We take only the 20 organisations with more relationships
cct=crosstab.sort_values(by='sum',ascending=False).iloc[:20]
cct


In [ ]:
### Specific periods with the residuals heatmap on the corrtable
cct.index=[e[:20]+'...' for e in cct.index]
pp = bl.plot_chi2_residuals(cct.iloc[:,:-1], figsize=(6,12))

### Employers

In [ ]:
### Create a cross-table for inspection
df=df_rel[df_rel['relationship']=='employment']
# Create the crosstab
crosstab = pd.crosstab(df['organisation_label'],df['per_activ'])

# Optional: Remove the index name if you want a cleaner look
crosstab.index.name = None
# crosstab.columns.name = None

crosstab=crosstab.fillna(0)
crosstab=crosstab.astype(int)
crosstab['sum'] = crosstab.sum(axis=1)

### Evolution of number of students in major universities (population in Wikidata)
print('Number of persons per organisation and period:', len(crosstab))
cct=crosstab.sort_values(by='sum',ascending=False).iloc[:20]
cct

In [ ]:
cct.index=[e[:25]+'...' for e in cct.index]
pp = bl.plot_chi2_residuals(cct.iloc[:,:-1], figsize=(6,12))

### Memberships

In [ ]:
### Create a cross-table for inspection
df=df_rel[df_rel['relationship']=='membership']
# Create the crosstab
crosstab = pd.crosstab(df['organisation_label'],df['per_activ'])

# Optional: Remove the index name if you want a cleaner look
crosstab.index.name = None
# crosstab.columns.name = None

crosstab=crosstab.fillna(0)
crosstab=crosstab.astype(int)
crosstab['sum'] = crosstab.sum(axis=1)

### Evolution of number of students in major universities (population in Wikidata)
print('Number of persons per organisation and period:', len(crosstab))
cct=crosstab.sort_values(by='sum',ascending=False).iloc[:20]
cct

In [ ]:
cct.index=[e[:25]+'...' for e in cct.index]
pp = bl.plot_chi2_residuals(cct.iloc[:,:-1], figsize=(6,10))

## Add years to approximate possible relations


In this exercice we artificially build relationships between persons without providing a measure of their real match to reality. In a heuristic perspective, the aim is just to have a consistent network in view of applying some analysis techniques.


We model artificial or imputed encounters based on the idea that persons present in the same institutions in the same period virtually know each other. But this is not alway the case although it allows to build person networks around organisations.

In [ ]:
### fonction that codes the year of begin of an 'encounter'
def code_year_begin(type_relation, year):
    if type_relation=='education':
        begin=int(year)+20
    elif type_relation=='employment':
        begin=int(year)+28
    elif type_relation=='membership':
        begin=int(year)+40
    else :
        begin = 0

    return begin    

In [ ]:
### function that codes the year of end of the virtual "encounter"
def code_year_end(type_relation, year):
    if type_relation=='education':
        end=int(year)+28
    elif type_relation=='employment':
        end=int(year)+65
    elif type_relation=='membership':
        end=int(year)+65
    else :
        end = 0
        
    return end

In [ ]:
### Apply the functions 
df_rel['year_begin'] = df_rel.apply(lambda x : code_year_begin(x['relationship'], x['birthYear']), axis=1)
df_rel['year_end'] = df_rel.apply(lambda x : code_year_end(x['relationship'], x['birthYear']), axis=1)

In [ ]:
### Inspect the coding
df_rel[df_rel.relationship=='membership'].iloc[10:15]

## Create pairs of persons: education

Create relations of persons through relations to organisations

In [ ]:
### Select the columns that will be used
df_sel = df_rel[df_rel.relationship=='education'][['person_uri','labelPer','organisation_uri', 'organisation_label', 'year_begin','year_end', 'per_activ']].copy(deep=True)
### Order by person_uri
df_sel=df_sel.sort_values(by='person_uri')
print(len(df_sel))

In [ ]:
### Join on common organisation: cartesian product -> produces a lot of rows !
merged_edu = pd.merge(df_sel, df_sel, on=['organisation_uri', 'organisation_label'])
print(len(merged_edu))

In [ ]:
### Inspect
merged_edu.head(2)

In [ ]:
### Eliminate double rows :relationship A-B but not relationship B-A
merged_edu = merged_edu[merged_edu['person_uri_x'] < (merged_edu['person_uri_y'])]
print(len(merged_edu), '-', len(merged_edu)*2)

In [ ]:
### Restrict time overlap (allow only 7 years overlap) — could be extended to have a larger result

# Note that many persons can be at the same time in the same university: this creates a number of relations

merged_edu = merged_edu[(merged_edu['year_begin_y'] < merged_edu['year_end_x']) & (merged_edu['year_begin_x'] < merged_edu['year_end_y'])]
print(len(merged_edu))

## We observe that linked people are not necessarily in the same activity period
merged_edu.head(3)


In [ ]:
### Number of educational relations per university 
gbe_edu = duckdb.query("""
    
    SELECT organisation_label, COUNT(*) as number
    FROM merged_edu
    GROUP BY organisation_label
    ORDER BY number DESC
    LIMIT 100
""").to_df()
gbe_edu.iloc[:10]

### Aggregate relationships during studies

#### Example of connected persons

Displayed with **Allegrograph Gruff**

<img src="images/alg_gruff_example_chinese_connected_education.png" alt="drawing" width="700"/>

In [ ]:
### Group by persons' pairs and count/aggregate organisations

## This is needed because the nx.add_edges_from() function applies
# a DISTINCT approach and information will be lost if two persons
# are related by more than one organisation -- this can virtually strengthen their relation
gr_edu = merged_edu.groupby(by=['person_uri_x', 'person_uri_y', 'per_activ_x', 'year_begin_x', 'year_end_x', 
                          'year_begin_y', 'year_end_y', 'per_activ_y']).agg({'organisation_label': '|'.join, 'organisation_uri': '|'.join})
gr_edu['number'] = gr_edu.organisation_label.apply(lambda x : len(x.split('|')))

print('Length Edu Network:', len(gr_edu))


In [ ]:
gr_edu=gr_edu.reset_index()
gr_edu.sort_values(by='number', ascending=False).head()

In [ ]:
### change columns names
gr_edu.columns=['person_uri_x', 'person_uri_y', 'per_activ_x', 'year_begin_x', 'year_end_x', 'year_begin_y', 'year_end_y', 'per_activ_y', 'orgs_labels', 'orgs_uris', 'orgs_number']
### change columns positions
gr_edu = gr_edu[['person_uri_x', 'person_uri_y', 'orgs_labels', 'orgs_number', 'orgs_uris', 'per_activ_x',  'per_activ_y']]
gr_edu.sort_values(by='orgs_number', ascending=False).head(2)

In [ ]:
### The overwhelming majority of the population has only one study relationship
print('More then 2 relations:', len(gr_edu[gr_edu.orgs_number>2]), '\n2 relations:', len(gr_edu[gr_edu.orgs_number==2]),
      '\none relation:', len(gr_edu[gr_edu.orgs_number==1]))

In [ ]:
### Find persons in different periods

# Related persons can belong to different periods

## We observe that there is significant overlap: 
## These people and pairs represent BRIDGES between periods !

print(len(gr_edu[gr_edu.per_activ_x != gr_edu.per_activ_y]))
gr_edu[gr_edu.per_activ_x != gr_edu.per_activ_y].sort_values(by='orgs_number', ascending=False).iloc[:5]

## Create pairs of persons: employment


Create relations of persons through relations to organisations

In [ ]:
### Select the columns that will be used
df_sel = df_rel[df_rel.relationship=='employment'][['person_uri','labelPer','organisation_uri', 'organisation_label', 'year_begin','year_end', 'per_activ']].copy(deep=True)
### Order by person_uri
# df_sel=df_sel.sort_values(by='person_uri')
print(len(df_sel))

In [ ]:
### Join on common organisation: cartesian product -> produces a lot of rows !
merged_empl = pd.merge(df_sel, df_sel, on=['organisation_uri', 'organisation_label'])
print(len(merged_empl))


In [ ]:
merged_empl.head(2)

In [ ]:
### Eliminate double rows :relationship A-B but not relationship B-A
merged_empl = merged_empl[merged_empl['person_uri_x'] < (merged_empl['person_uri_y'])]
print(len(merged_empl), '-', len(merged_empl)*2)

In [ ]:
### Restrict time overlap : from 28 to 65
# Note that many persons can be at the same time in the same university: this creates a number of relations

merged_empl = merged_empl[(merged_empl['year_begin_y'] < merged_empl['year_end_x']) & (merged_empl['year_begin_x'] < merged_empl['year_end_y'])]
print(len(merged_empl))

## We observe that linked people are not necessarily in the same activity period
merged_empl.head(3)


In [ ]:
### Organisations with most frequent co-employments
gbe_empl = duckdb.query("""
    SELECT organisation_label, COUNT(*) as number
    FROM merged_empl
    GROUP BY organisation_label
    ORDER BY number DESC
    LIMIT 100
""").to_df()
gbe_empl.iloc[:10]

### Create network of persons relationships during emloyment

In [ ]:
### Group by persons' pairs and count/aggregate organisations

## This is needed because the nx.add_edges_from() function applies
# a DISTINCT approach and information will be lost if two persons
# are related by more than one organisation -- this can virtually strengthen their relation
gr_empl = merged_empl.groupby(by=['person_uri_x', 'person_uri_y', 'per_activ_x', 'year_begin_x', 'year_end_x', 
                          'year_begin_y', 'year_end_y', 'per_activ_y']).agg({'organisation_label': '|'.join, 'organisation_uri': '|'.join})
gr_empl['number'] = gr_empl.organisation_label.apply(lambda x : len(x.split('|')))

print(len(gr_empl))


In [ ]:
gr_empl=gr_empl.reset_index()
gr_empl.sort_values(by='number', ascending=False).head(2)

In [ ]:
### change columns names
gr_empl.columns=['person_uri_x', 'person_uri_y', 'per_activ_x', 'year_begin_x', 'year_end_x', 'year_begin_y', 'year_end_y', 'per_activ_y', 'orgs_labels', 'orgs_uris', 'orgs_number']
### change columns positions
gr_empl = gr_empl[['person_uri_x', 'person_uri_y', 'orgs_labels', 'orgs_number', 'orgs_uris', 'per_activ_x', 'per_activ_y']]
gr_empl.sort_values(by='orgs_number', ascending=False).head(2)

In [ ]:
### The overwhelming majority of the population has only one occupation relationship
print('More then 2 relations:', len(gr_empl[gr_empl.orgs_number>2]), 
      '\n2 relations:', len(gr_empl[gr_empl.orgs_number==2]),
      '\n1 relations:', len(gr_empl[gr_empl.orgs_number==1]))
print(f'Maximum number of relationships in employments: {gr_empl.orgs_number.max()}')

## Create pairs of persons: membership

Create relations of persons through relations to organisations

In [ ]:
### Select the columns that will be used
df_sel = df_rel[df_rel.relationship=='membership'][['person_uri','labelPer','organisation_uri', 'organisation_label', 'year_begin','year_end', 'per_activ']].copy(deep=True)
### Order by person_uri
df_sel=df_sel.sort_values(by='person_uri')
print(len(df_sel))

In [ ]:
### Join on common organisation: cartesian product -> produces a lot of rows !
merged_memb = pd.merge(df_sel, df_sel, on=['organisation_uri', 'organisation_label'])
print(len(merged_memb))


In [ ]:
### Eliminate double rows :relationship A-B but not relationship B-A
merged_memb = merged_memb[merged_memb['person_uri_x'] < (merged_memb['person_uri_y'])]
print(len(merged_memb), '-', len(merged_memb)*2)

In [ ]:
### Restrict time overlap (allow only 7 years overlap) — could be extended to have a larger result

# Note that many persons can be at the same time in the same university: this creates a number of relations

merged_memb = merged_memb[(merged_memb['year_begin_y'] < merged_memb['year_end_x']) & (merged_memb['year_begin_x'] < merged_memb['year_end_y'])]
print(len(merged_memb))

## We observe that linked people are not necessarily in the same activity period
merged_memb.head(3)


In [ ]:
### count membership relations by organisation
gbe_memb=duckdb.query("""
    SELECT organisation_label, COUNT(*) as number
    FROM merged_memb
    GROUP BY organisation_label
    ORDER BY number DESC
    LIMIT 100
""").to_df()
gbe_memb.iloc[:10]

### Create network of persons relationships during membership

In [ ]:
### Group by persons' pairs and count/aggregate organisations

## This is needed because the nx.add_edges_from() function applies
# a DISTINCT approach and information will be lost if two persons
# are related by more than one organisation -- this can virtually strengthen their relation
gr_memb = merged_memb.groupby(by=['person_uri_x', 'person_uri_y', 'per_activ_x', 'year_begin_x', 'year_end_x', 
                          'year_begin_y', 'year_end_y', 'per_activ_y']).agg({'organisation_label': '|'.join, 'organisation_uri': '|'.join})
gr_memb['number'] = gr_memb.organisation_label.apply(lambda x : len(x.split('|')))
gr_memb=gr_memb.reset_index()
print(len(gr_memb))


In [ ]:
### Inspect grouped membership relationships
gr_memb.sort_values(by='number', ascending=False).head()

In [ ]:
### change columns names
gr_memb.columns=['person_uri_x', 'person_uri_y', 'per_activ_x', 'year_begin_x', 'year_end_x', 'year_begin_y', 'year_end_y', 'per_activ_y', 'orgs_labels', 'orgs_uris', 'orgs_number']

In [ ]:
### change columns positions
gr_memb = gr_memb[['person_uri_x', 'person_uri_y', 'orgs_labels', 'orgs_number', 'orgs_uris', 'per_activ_x', 'per_activ_y']]
gr_memb.sort_values(by='orgs_number', ascending=False).head(2)

In [ ]:
print('More then 2 relations:', len(gr_memb[gr_memb.orgs_number>2]), 
      '\n2 relations:', len(gr_memb[gr_memb.orgs_number==2]),
      '\none relation:', len(gr_memb[gr_memb.orgs_number==1]))
print(f'Maximum number of relations in memberships: {gr_memb.orgs_number.max()}')

In [ ]:
print('More then 4 relations:', len(gr_memb[gr_memb.orgs_number>4]), 
      '\n4 relations:', len(gr_memb[gr_memb.orgs_number==4]),
      '\n3 relations:', len(gr_memb[gr_memb.orgs_number==3]))

In [ ]:
print('More then 10 relations:', len(gr_memb[gr_memb.orgs_number>10]), '\n10 relations:', len(gr_memb[gr_memb.orgs_number==10]))

In [ ]:
# Plot the distribution of eigenvector density

df=gr_memb[gr_memb.orgs_number>5]

min_val = df.orgs_number.min()
max_val = df.orgs_number.max()

plt.figure(figsize=(12, 3))
ax = sns.violinplot(data=df.orgs_number, orient='h')

# Set the y-axis limits to exactly match the data range
ax.set_xlim(min_val, max_val)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.title('Distribution (density) of relations of persons to organisations')
plt.show()

## Most frequent organisations as relations

Organisations where memberships' relationship happen are different from those where people are educated and are employed, as we can see if we compare the tables below.

We therefore just join educational institutions and employers

In [ ]:
### Study
gbe_edu.head(10)

In [ ]:
### Employment
gbe_empl.head(10)

In [ ]:
### Join first hundred organisation for education and employment

gb_organ_of_rel=pd.merge(left=gbe_edu, on='organisation_label', right=gbe_empl,
                suffixes=('_rel_edu', '_rel_empl'), how='outer')
gb_organ_of_rel[['number_rel_edu','number_rel_empl']]=gb_organ_of_rel[['number_rel_edu','number_rel_empl']].fillna(0).astype('int')
print(len(gb_organ_of_rel))
gb_organ_of_rel.head()

In [ ]:
### Inspect most frequent organisations, order by study
gb_organ_of_rel.sort_values(by='number_rel_edu', ascending=False).iloc[:10]

In [ ]:
### Inspect most frequent organisations, order by employer
# Note the difference, e.g. the MSU Faculty of Physcs has no employee ... 
# There should be a bias in the data (not yet verified)
# See also CERN and Bell Labs
gb_organ_of_rel.sort_values(by='number_rel_empl', ascending=False).iloc[:30]


In [ ]:
### Membership
# These are totally different institution with regard to the former ones
gbe_memb.head(10)

## Merge the three networks: where do the encounters happen?

We merge the three tables of relations (education, employment, membership) and observe where encounters happen.

In [ ]:
### Join educational and employment persons' pairs
# All relationships are kept, left and right join -> outer
network=pd.merge(gr_edu, gr_empl, on=['person_uri_x', 'person_uri_y', 'per_activ_x', 'per_activ_y'],
                 how='outer', suffixes=('_edu', '_empl'), indicator=True)
network.rename(columns={'_merge': 'merge_1'}, inplace=True)

In [ ]:
print('Number of edu+empl pairs:',len(network))
network.sort_values(by='orgs_number_edu', ascending=False).head(2)

In [ ]:
### Join to network the membership persons' pairs
# All relationships are kept, left and right join -> outer
network=pd.merge(network, gr_memb, on=['person_uri_x', 'person_uri_y', 'per_activ_x', 'per_activ_y' ],
                 how='outer', suffixes=('', '_memb'), indicator=True)
network.rename(columns={'orgs_labels': 'orgs_labels_memb','orgs_number': 'orgs_number_memb','organisation_uri': 'orgs_uris_memb'}, inplace=True)

In [ ]:
network['orgs_number_edu'].fillna(0, inplace=True)
network['orgs_number_empl'].fillna(0, inplace=True)
network['orgs_number_memb'].fillna(0, inplace=True)
network['orgs_number_memb']=network['orgs_number_memb'].astype(int)
network['orgs_number_edu']=network['orgs_number_edu'].astype(int)
network['orgs_number_empl']=network['orgs_number_empl'].astype(int)
network=network[['person_uri_x', 'person_uri_y', 'orgs_labels_edu', 'orgs_number_edu', 'per_activ_x', 'per_activ_y', 'orgs_labels_empl', 'orgs_number_empl', 'orgs_labels_memb', 'orgs_number_memb']]

In [ ]:
print('Number of edu+empl+memb pairs:',len(network))
network.iloc[550:555]

In [ ]:
### create field with sum of edu and empl
network['sum_edu_empl']=network['orgs_number_edu']+network['orgs_number_empl']

In [ ]:
### create field with sum of all values
network['sum_all_rel']=network['orgs_number_edu']+network['orgs_number_empl']+network['orgs_number_memb']

#### Add persons' labels

In [ ]:
network=pd.merge(left=network,right=df_p[['person_uri','labelPer','birthYear']], left_on='person_uri_x', right_on='person_uri')

In [ ]:
network.drop(columns=['person_uri'], inplace=True)

In [ ]:
network=pd.merge(left=network,right=df_p[['person_uri','labelPer','birthYear']], left_on='person_uri_y', right_on='person_uri', suffixes=('_x1', '_y'))


In [ ]:
network.head(2)

In [ ]:
network.rename(columns={'labelPer_x1': 'labelPer_x','birthYear_x1': 'birthYear_x'}, inplace=True)

In [ ]:
network.drop(columns=['person_uri'], inplace=True)

In [ ]:
network.head(2)

In [ ]:
### Change columns order
network=network[['labelPer_x',
  'labelPer_y',
  'birthYear_x','birthYear_y',
'per_activ_x', 'per_activ_y',
 'sum_edu_empl', ### study and employment relationshipa
 'sum_all_rel', ### all the relationships
 'orgs_number_edu',  'orgs_labels_edu', 
 'orgs_number_empl', 'orgs_labels_empl',
 'orgs_number_memb', 'orgs_labels_memb',
 'person_uri_x', 'person_uri_y'
   ]]

In [ ]:
### SQL Queries


# order by max sum of edu and empl relations
query_1="""
    SELECT *
    FROM network
    ORDER BY sum_edu_empl DESC
    LIMIT 5
"""


# order by max sum of all relations
query_2="""
    SELECT * 
    FROM network
    WHERE orgs_number_edu > 0
    AND orgs_number_empl>0
    ORDER BY sum_all_rel DESC
    --OFFSET 500
    LIMIT 10
"""

# order by max sum of all relations
query_2_count="""
    SELECT COUNT(*) as number  -- 3922
    FROM network
    WHERE orgs_number_edu > 0
    AND orgs_number_empl>0
"""

# order by max  edu and empl relations
query_3="""
    SELECT *
    FROM network
    WHERE orgs_number_edu > 0
    AND orgs_number_empl>0
    ORDER BY orgs_number_empl DESC, orgs_number_edu DESC
    --OFFSET 500
    LIMIT 10
"""

In [ ]:
duckdb.query(query_2).to_df()

In [ ]:
print('Tot:', len(network), 'Edu:', len
      (gr_edu), ', Empl:', len(gr_empl), ', Memb:', len(gr_memb))

### Store the network on the disk

DO NOT COMMIT THIS FILE IF NOT ADDED *.pkl  TO .gitignore

In [ ]:
### BEWARE: add all pickle files to the .gitignore file with this entry . *.pkl

### DO NOT COMMIT THIS FILE IF NOT ADDED *.pkl  TO .gitignore

file_address='da_data/da6-networks.pkl'
network.to_pickle(file_address)

## Correlations

We could wonder if the fact of studying together, or working together would lead to be member of the same organisations.

Given the large amount of people having just one or to relation, we only consider people having at least four relationships.

We use here ranking correlation given that we are counting relations

Surprisingly:
* there is some weak positive relation between studying and working together
* but there is a stronger negative correlation between studying or working together and being member of same organisations 


In [ ]:
### ONLY EXECUTE if you need to restart the notebook from the stored network

file_address='da_data/da6-networks.pkl'
network=pd.read_pickle(file_address)
print(len(network))
network.iloc[:3]

In [ ]:
### Inspect cooccurrence of pairs, before analysing correlations

# We observe that we have quite sparse data in the network !

print('Total number of relationships:',len(network))
print('Edu', len(network[(network.orgs_number_edu>0)]))
print('Empl', len(network[(network.orgs_number_empl>0)]))
print('Memb', len(network[(network.orgs_number_memb>0)]))
print('Study and work together',len(network[(network.orgs_number_edu>0) & (network.orgs_number_empl>0)]))
print('Study and membership together', len(network[(network.orgs_number_edu>0) & (network.orgs_number_memb>0) ]))
print('Work and membership together',len(network[ (network.orgs_number_empl>0) & (network.orgs_number_memb>0) ]))
print('All three phases together',len(network[(network.orgs_number_edu>0) & (network.orgs_number_empl>0) & (network.orgs_number_memb>0) ]))

In [ ]:
### We only consider in the distribution more than three
df=network[network.sum_all_rel>5].copy(deep=True)
print(len(df))
print(df.sum_all_rel.describe())

In [ ]:

### Plot the distribution of the total number of relations
# where there are more than 3 relations
min_val = df.sum_all_rel.min()
max_val = df.sum_all_rel.max()

plt.figure(figsize=(12, 3))
ax = sns.violinplot(data=df.sum_all_rel, orient='h')

# Set the y-axis limits to exactly match the data range
ax.set_xlim(min_val, max_val)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.title('Distribution (density) of relations of persons to organisations')
plt.show()

### Measure the correlations

#### Relationships in studies and in employment

We only can use rank correlation given these are counts. 

Also, most educational relations have count: 1 - so the correlation does not really make sense

In [ ]:
predictor_name='orgs_number_edu' # 'orgs_number_edu' 'orgs_number_empl'
dependent_name='orgs_number_empl'

# Prepare variables
X = df[predictor_name]
Y = df[dependent_name]

# 1. Correlation Analysis: on ranking because these are counts
spearman_corr, p_val_spearman = spearmanr(X, Y)

print(f"Spearman Correlation: {spearman_corr:.4f} (p-value: {p_val_spearman:.4f})")


#### Relationships in studies and in membership

In [ ]:

predictor_name='orgs_number_edu' # 'orgs_number_edu' 'orgs_number_empl'
dependent_name='orgs_number_memb'

# Prepare variables
X = df[predictor_name]
Y = df[dependent_name]

# 1. Correlation Analysis: on ranking because these are counts
spearman_corr, p_val_spearman = spearmanr(X, Y)

print(f"Spearman Correlation: {spearman_corr:.4f} (p-value: {p_val_spearman:.4f})")


#### Relationships in employment and in membership

In [ ]:

predictor_name='orgs_number_empl' # 'orgs_number_edu' 'orgs_number_empl'
dependent_name='orgs_number_memb'

# Prepare variables
X = df[predictor_name]
Y = df[dependent_name]

# 1. Correlation Analysis
spearman_corr, p_val_spearman = spearmanr(X, Y)

print(f"Spearman Correlation: {spearman_corr:.4f} (p-value: {p_val_spearman:.4f})")


### Represent the correlations

In [ ]:
df=df[['sum_all_rel', 'orgs_number_edu', 
       'orgs_number_empl', 'orgs_number_memb']].\
              sort_values(by=['orgs_number_memb'])
df.reset_index(inplace=True, drop=True)
df.head()

In [ ]:
# Smooth with gliding mean

# Ensure data is sorted by index (and value)

df['smoothed_memb'] = df['orgs_number_memb'].rolling(window=100, min_periods=1).mean()
df['smoothed_empl'] = df['orgs_number_empl'].rolling(window=100, min_periods=5).mean()
df['smoothed_edu'] = df['orgs_number_edu'].rolling(window=100, min_periods=1).mean()
df['smoothed_all'] = df['sum_all_rel'].rolling(window=100, min_periods=1).mean()



In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.plot(df['smoothed_empl'], linestyle='-', color='red', linewidth=1, label='Trend Empl')
ax.plot(df['smoothed_memb'],  linestyle='-', color='blue', linewidth=1, label='Trend Memb')
ax.plot(df['smoothed_edu'],  linestyle='-', color='black', linewidth=1, label='Trend Edu')
ax.plot(df['smoothed_all'],  linestyle='-', color='brown', linewidth=2, label='Trend Total')

ax.set_xlabel('Persons pairs')
ax.set_ylabel('Value')
ax.set_title('Line Chart with Smoothed Values (oredered by memberships)')
ax.grid(True, linestyle='--')
ax.legend()
plt.tight_layout()
plt.show()